# Workshop Schedule and Sign-up

In [10]:
import pandas as pd
from IPython.display import display, HTML
from datetime import datetime
from html import escape

# Load data
file_path = '../data/workshop_info.csv'
courses_df = pd.read_csv(file_path)
style_path = '../data/course_style.csv'
style_df = pd.read_csv(style_path)

# Clean headers
courses_df.columns = courses_df.columns.str.strip()
style_df.columns = style_df.columns.str.strip()

# Build course style lookup
style_map = {}
if {'course_name', 'course_style'}.issubset(style_df.columns):
    style_df['course_name'] = style_df['course_name'].fillna('').astype(str).str.strip()
    style_df['course_style'] = style_df['course_style'].fillna('').astype(str).str.strip()
    style_map = dict(zip(style_df['course_name'], style_df['course_style']))

# Badge colors
badge_styles = {
    'Live Coding': ('#0747a6', '#e9f2ff'),
    'Seminar': ('#7a4f01', '#fff2db'),
    'Hybrid': ('#5d2d86', '#f4ebff'),
    'Surgery': ('#0f766e', '#e6fffa'),
    'Self-Paced': ('#1f6f43', '#e8f8ef'),
}

# Parse dates
courses_df['End Date'] = pd.to_datetime(courses_df['End Date'], dayfirst=True, errors='coerce')
courses_df['Start Date'] = pd.to_datetime(courses_df['Start Date'], dayfirst=True, errors='coerce')

today = datetime.now()

# Filter upcoming/current courses
mask_dates = courses_df['Course Date'].notna()
mask_started = courses_df['Start Date'] <= today
mask_not_ended = courses_df['End Date'] > today

upcoming = (
    courses_df[mask_dates & mask_started & mask_not_ended]
    .sort_values('End Date')
    .reset_index(drop=True)
)

html_header = """
<table style="width:100%; text-align:center; border:1px solid black; border-collapse:collapse;">
  <tr style="background:#f2f2f2;">
    <th>Course Name</th>
    <th>Course Dates</th>
    <th>Location</th>
    <th>Registration Link</th>
    <th>Course Style</th>
  </tr>
"""

def generate_row(row):
    name = row['Course Name']

    # Course name link to workshop page
    page_url = row.get('Homepage Filepath', '')
    if pd.notna(page_url) and str(page_url).strip():
        name_html = f'<a href="{page_url}" target="_blank">{escape(str(name))}</a>'
    else:
        name_html = escape(str(name))

    # Style badge in dedicated column
    style = style_map.get(str(name).strip(), '')
    if isinstance(style, str) and style.strip():
        style_label = style.strip()
        text_color, bg_color = badge_styles.get(style_label, ('#3f3f46', '#f4f4f5'))
        style_html = (
            f"<a href='course_styles.html' target='_blank' title='Read about course delivery styles' style='text-decoration:none;'>"
            f"<span style='display:inline-block; padding:0.24rem 0.72rem; border-radius:999px; border:2px solid {text_color}; color:{text_color}; background:{bg_color}; font-size:0.86rem; font-weight:700;'>"
            f"{escape(style_label)}"
            "</span></a>"
        )
    else:
        style_html = ""

    dates = row['Course Date']
    location = row.get('Location', '')

    signup = row.get('MS Form Signup', '')
    if pd.isna(signup) or not str(signup).strip():
        link_html = "Registration Will Be Available Soon"
    else:
        link_html = f'<a href="{signup}" target="_blank">Sign-Up</a>'

    return (
        "<tr>"
        f"<td style='padding:8px'>{name_html}</td>"
        f"<td style='padding:8px'>{dates}</td>"
        f"<td style='padding:8px'>{location}</td>"
        f"<td style='padding:8px'>{link_html}</td>"
        f"<td style='padding:8px'>{style_html}</td>"
        "</tr>"
    )

rows_html = "\n".join(upcoming.apply(generate_row, axis=1).tolist())
html_table = html_header + rows_html + "\n</table>"

display(HTML(html_table))


Course Name,Course Dates,Location,Registration Link
Introduction to Julia,2nd/9th/16th June 2025 1-4pm,Online only,Sign-Up
Introduction to HPC,4th/11th June 2025 1-4pm,In-person only,Sign-Up
Introduction to Markdown in R,6th June 2025 10am-1pm,In-person only,Sign-Up
Using Markdown for Python,10th June 2025 10am-1pm,In-person only,Sign-Up
Parallel Computing,12th/19th June 2025 10am-1pm,Online only,Sign-Up


# Previous Workshops

In [1]:
import pandas as pd
from IPython.display import display, HTML

# Load the CSV file (adjust the file path as needed)
file_path_previous = '../data/previous_workshops.csv'
previous_courses_df = pd.read_csv(file_path_previous)
previous_courses_df = previous_courses_df.iloc[::-1].reset_index(drop=True)

# Strip any extra spaces in the column names
previous_courses_df.columns = previous_courses_df.columns.str.strip()

# Function to generate an HTML table row for each course
def generate_html_row(row):
    #return f"<tr><td>{row['Course Name']}</td><td>{row['Course Info']}</td><td>{row['Course Leader']}</td><td>{row['Course Helpers']}</td><td>{row['Course Developers']}</td></tr>"
    return f"<tr><td>{row['Course Name']}</td><td>{row['Course Info']}</td><td>{row['Course Leader']}</td><td>{row['Course Helpers']}</td></tr>"

#Generate HTML table header
# html_table_header = """
# <table style='width: 100%; text-align: center; border: 1px solid black; border-collapse: collapse;'>
# <tr>
# <th>Course Name</th>
# <th>Course Info</th>
# <th>Course Leader</th>
# <th>Course Helpers</th>
# <th>Course Developers</th>
# </tr>
# """
html_table_header = """
<table style='width: 100%; text-align: center; border: 1px solid black; border-collapse: collapse;'>
<tr>
<th>Course Name</th>
<th>Course Date</th>
<th>Course Leader</th>
<th>Course Helpers</th>
</tr>
"""

# Apply the function and create the HTML table rows
table_rows = previous_courses_df.apply(generate_html_row, axis=1).tolist()

# Join the table rows into a single block
html_table = html_table_header + "\n".join(table_rows) + "</table>"

# Display the HTML table
display(HTML(html_table))


Course Name,Course Date,Course Leader,Course Helpers
Parallel Computing,Jun-25,Ricky Olivier,Liam Berrisford
Introduction to HPC&ISCA,Jun-25,Michael Saunby,Jingzhan Lu
Using Markdown for R,Jun-25,Craig Willis,Conor Crilly
Introduction to Julia,Jun-25,Tom Hawes,"Liam Berrisford, Jingzhan Lu"
Introduction to R,May-25,Stephen Lang,Ishaan Sinha
Introduction to Python,May-25,Neil Vaughan,"Tom Hawes, Jeremy Pike, Catherine Russon"
Introduction to version control,May-25,Ned Taylor,"Aditi Dutta, John Luff"
Improve Your R Code,Apr-25,Conor Crilly,"Michelle Ledbetter, Craig Willis"
Working with data in R,Apr-25,Tim Fawcett,"Michelle Ledbetter, Bertrand Nortier, Cuihong Xie"
Regression Analysis with R,Mar-25,Robin Beaumont,"Theresa Wacker, Philippa Wells"
